In [0]:
dbutils.widgets.text("catalog_name", "allianz_coe", "Catalog Name")
dbutils.widgets.text("gold_schema_name", "gold", "Gold Schema Name")
dbutils.widgets.text("vault_schema_name", "silver", "Vault Schema Name")

catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
vault_schema = dbutils.widgets.get("vault_schema_name")

In [0]:
DIM_CAMPAIGN_CFG = {
    "name": "dim_campaign",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_campaign",
    "business_key_col": "campaign_id",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
        WITH src AS (
  SELECT
    hu.campaign_id AS campaign_id,
    sa.campaign_name AS campaign_name,
    sa.campaign_type AS campaign_type,
    sa.campaign_start_date AS campaign_start_date,
    sa.campaign_end_date AS campaign_end_date,
    sa.campaign_status AS campaign_status,
    sa.campaign_budget AS campaign_budget,
    sa.campaign_target_audience AS campaign_target_audience,
    sa.campaign_marketing_source AS campaign_marketing_source,
    sa.campaign_owner_department AS campaign_owner_department,
    sa.campaign_country AS campaign_country,
    sa.campaign_conversion_goal AS campaign_conversion_goal,
    sa.number_of_clicks AS number_of_clicks,
    sa.is_active AS is_active,
    sa.number_of_visits AS number_of_visits,
    sa.number_of_policy_purchases AS number_of_policy_purchases,
    sa.number_of_emails_sent AS number_of_emails_sent,
    sa.number_of_email_bounced AS number_of_email_bounced,
    sa.number_of_emails_delivered AS number_of_emails_delivered,
    sa.number_of_emails_opened AS number_of_emails_opened,
    sa.click_through_rate AS click_through_rate,
    sa.spend_amount AS spend_amt,
    sa.incremental_revenue AS incremental_revenue,
    sa.survey_wave AS survey_wave,
    sa.total_number_of_respondents AS total_number_of_respondents,
    sa.number_of_respondents_aware AS number_of_respondents_aware,
    sa.number_of_promoters AS number_of_promoters,
    sa.number_of_passives AS number_of_passives,
    sa.number_of_detractors AS number_of_detractors,
    sa.number_of_followers AS number_of_followers,
    sa.number_of_likes AS number_of_likes,
    sa.number_of_comments AS number_of_comments,
    sa.number_of_shares AS number_of_shares,
    sa.number_of_brand_mentions AS number_of_brand_mentions,
    sa.number_of_category_mentions AS number_of_category_mentions,
    sa.number_of_impressions AS number_of_impressions,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.HUB_CAMPAIGN hu
  LEFT JOIN {catalog_name}.{vault_schema}.SAT_CAMPAIGN sa
    ON hu.campaign_hash_key = sa.campaign_hash_key
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    campaign_start_date,
    campaign_end_date,
    campaign_status,
    campaign_budget,
    campaign_target_audience,
    campaign_marketing_source,
    campaign_owner_department,
    campaign_country,
    campaign_conversion_goal,
    number_of_clicks,
    is_active,
    number_of_visits,
    number_of_policy_purchases,
    number_of_emails_sent,
    number_of_email_bounced,
    number_of_emails_delivered,
    number_of_emails_opened,
    click_through_rate,
    spend_amt,
    incremental_revenue,
    survey_wave,
    total_number_of_respondents,
    number_of_respondents_aware,
    number_of_promoters,
    number_of_passives,
    number_of_detractors,
    number_of_followers,
    number_of_likes,
    number_of_comments,
    number_of_shares,
    number_of_brand_mentions,
    number_of_category_mentions,
    number_of_impressions,
    ROW_NUMBER() OVER (
      PARTITION BY campaign_id
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  campaign_id,
  campaign_name,
  campaign_type,
  campaign_start_date,
  campaign_end_date,
  campaign_status,
  campaign_budget,
  campaign_target_audience,
  campaign_marketing_source,
  campaign_owner_department,
  campaign_country,
  campaign_conversion_goal,
  number_of_clicks,
  is_active,
  number_of_visits,
  number_of_policy_purchases,
  number_of_emails_sent,
  number_of_email_bounced,
  number_of_emails_delivered,
  number_of_emails_opened,
  click_through_rate,
  spend_amt,
  incremental_revenue,
  survey_wave,
  total_number_of_respondents,
  number_of_respondents_aware,
  number_of_promoters,
  number_of_passives,
  number_of_detractors,
  number_of_followers,
  number_of_likes,
  number_of_comments,
  number_of_shares,
  number_of_brand_mentions,
  number_of_category_mentions,
  number_of_impressions,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["campaign_id", "campaign_name", "campaign_type", "campaign_start_date", "campaign_end_date", "campaign_status", "campaign_budget", "campaign_target_audience", "campaign_marketing_source", "campaign_owner_department", "campaign_country", "campaign_conversion_goal", "number_of_clicks", "is_active", "number_of_visits", "number_of_policy_purchases", "number_of_emails_sent", "number_of_email_bounced", "number_of_emails_delivered", "number_of_emails_opened", "click_through_rate", "spend_amt", "incremental_revenue", "survey_wave", "total_number_of_respondents", "number_of_respondents_aware", "number_of_promoters", "number_of_passives", "number_of_detractors", "number_of_followers", "number_of_likes", "number_of_comments", "number_of_shares", "number_of_brand_mentions", "number_of_category_mentions", "number_of_impressions"
    ],

    "insert_cols": [
        "campaign_id", "campaign_name", "campaign_type", "campaign_start_date", "campaign_end_date", "campaign_status", "campaign_budget", "campaign_target_audience", "campaign_marketing_source", "campaign_owner_department", "campaign_country", "campaign_conversion_goal", "number_of_clicks", "is_active", "number_of_visits", "number_of_policy_purchases", "number_of_emails_sent", "number_of_email_bounced", "number_of_emails_delivered", "number_of_emails_opened", "click_through_rate", "spend_amt", "incremental_revenue", "survey_wave", "total_number_of_respondents", "number_of_respondents_aware", "number_of_promoters", "number_of_passives", "number_of_detractors", "number_of_followers", "number_of_likes", "number_of_comments", "number_of_shares", "number_of_brand_mentions", "number_of_category_mentions", "number_of_impressions",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_CHANNEL_CFG = {
    "name": "dim_channel",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_channel",
    "business_key_col": "channel_id",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
        WITH src AS (
  SELECT
    hu.channel_id AS channel_id,
    sa.channel_name AS channel_name,
    sa.channel_type AS channel_type,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.HUB_CHANNEL hu
  LEFT JOIN {catalog_name}.{vault_schema}.SAT_CHANNEL sa
    ON hu.channel_hash_key = sa.channel_hash_key
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    channel_id,
    channel_name,
    channel_type,
    ROW_NUMBER() OVER (
      PARTITION BY channel_id
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  channel_id,
  channel_name,
  channel_type,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["channel_id", "channel_name", "channel_type"
    ],

    "insert_cols": [
        "channel_id", "channel_name", "channel_type",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_GEOGRAPHY_CFG = {
    "name": "dim_geography",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_geography",
    "business_key_cols": ["country","state","city"],
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
        WITH src AS (
  SELECT
    sa.CITY AS city,
    sa.STATE AS state,
    sa.COUNTRY AS country,
	CASE
                WHEN UPPER(COUNTRY) IN ('USA','UNITED STATES','CANADA') THEN 'North America'
                WHEN UPPER(COUNTRY) IN ('INDIA') THEN 'Asia'
                WHEN UPPER(COUNTRY) IN ('UK','UNITED KINGDOM','GERMANY','FRANCE','SPAIN','ITALY') THEN 'Europe'
                WHEN UPPER(COUNTRY) IN ('AUSTRALIA','NEW ZEALAND') THEN 'Oceania'
                WHEN UPPER(COUNTRY) IN ('BRAZIL','ARGENTINA','CHILE','MEXICO') THEN 'Latin America'
                WHEN UPPER(COUNTRY) IN ('SOUTH AFRICA','NIGERIA','KENYA','EGYPT') THEN 'Africa'
                ELSE 'Unknown'
            END AS REGION,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.SAT_ADDRESS sa
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    city,
    state,
    country,
    region,
    ROW_NUMBER() OVER (
      PARTITION BY country,state,city
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  city,
  state,
  country,
  region,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["city", "state", "country", "region"
    ],

    "insert_cols": [
        "city", "state", "country", "region",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_INSURED_OBJECT_CFG = {
    "name": "dim_insured_object",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_insured_object",
    "business_key_col": "insured_object_id",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
        WITH src AS (
  SELECT
    hu.INSURED_OBJECT_ID AS insured_object_id,
    sa.INSURED_OBJECT_TYPE AS insured_object_type,
    sa.INSURED_OBJECT_SUB_TYPE AS insured_object_sub_type,
    sa.INSURED_OBJECT_DESCRIPTION AS insured_object_desc,
    sa.INSURED_VALUE AS insured_value,
    sa.CURRENCY_CODE AS currency_code,
    sa.INSURED_OBJECT_START_DATE AS insured_object_start_date,
    sa.INSURED_OBJECT_END_DATE AS insured_object_end_date,
    sa.INSURED_OBJECT_CURRENT_STATUS AS insured_object_current_status,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.HUB_INSURED_OBJECT hu
  LEFT JOIN {catalog_name}.{vault_schema}.SAT_INSURED_OBJECT sa
    ON hu.insured_object_hash_key = sa.insured_object_hash_key
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    insured_object_id,
    insured_object_type,
    insured_object_sub_type,
    insured_object_desc,
    insured_value,
    currency_code,
    insured_object_start_date,
    insured_object_end_date,
    insured_object_current_status,
    ROW_NUMBER() OVER (
      PARTITION BY insured_object_id
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  insured_object_id,
  insured_object_type,
  insured_object_sub_type,
  insured_object_desc,
  insured_value,
  currency_code,
  insured_object_start_date,
  insured_object_end_date,
  insured_object_current_status,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["insured_object_id", "insured_object_type", "insured_object_sub_type", "insured_object_desc", "insured_value", "currency_code", "insured_object_start_date", "insured_object_end_date", "insured_object_current_status"
    ],

    "insert_cols": [
        "insured_object_id", "insured_object_type", "insured_object_sub_type", "insured_object_desc", "insured_value", "currency_code", "insured_object_start_date", "insured_object_end_date", "insured_object_current_status",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_OVERRIDE_CFG = {
    "name": "dim_override",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_override",
    "business_key_col": "override_id",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
        WITH src AS (
  SELECT
    hu.override_id AS override_id,
    sa.override_reason AS override_reason,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.HUB_OVERRIDE hu
  LEFT JOIN {catalog_name}.{vault_schema}.SAT_OVERRIDE sa
    ON sa.override_hash_key = hu.override_hash_key
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    override_id,
    override_reason,
    ROW_NUMBER() OVER (
      PARTITION BY override_id
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  override_id,
  override_reason,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["override_id", "override_reason"
    ],

    "insert_cols": [
        "override_id", 
        "override_reason",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_REGULATION_CFG = {
    "name": "dim_regulation",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_regulation",
    "business_key_col": "regulation_id",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",
    "scd_type": "1",

    "stage_sql": f"""
       WITH src AS (
  SELECT
    hu.REGULATION_ID AS regulation_id,
    sa.REGULATION_NUMBER AS regulation_number,
    sa.REGULATION_NAME AS regulation_name,
    sa.REGULATION_DEPARTMENT AS regulation_department,
    sa.REGULATION_REGION AS regulation_region,
    sa.REGULATION_RISK_LEVEL AS regulation_risk_level,
    sa.REGULATION_COMPLIANCE_STATUS AS regulation_compliance_status,
    sa.REGULATION_DATE_RAISED AS regulation_date_raised,
    sa.REGULATION_DATE_CLOSED AS regulation_date_closed,
    sa.REGULATION_OWNER AS regulation_owner,
    sa.REGULATION_DEADLINE_DATE AS regulation_deadline_date,
    sa.IS_REGULATION_ON_TIME AS is_regulation_on_time,
    sa.load_date AS watermark_col
  FROM {catalog_name}.{vault_schema}.HUB_REGULATION hu
  LEFT JOIN {catalog_name}.{vault_schema}.SAT_REGULATION sa
    ON hu.regulation_hash_key = sa.regulation_hash_key
  WHERE sa.load_date > ${{watermark_col}}
),
dedup AS (
  SELECT
    regulation_id,
    regulation_number,
    regulation_name,
    regulation_department,
    regulation_region,
    regulation_risk_level,
    regulation_compliance_status,
    regulation_date_raised,
    regulation_date_closed,
    regulation_owner,
    regulation_deadline_date,
    is_regulation_on_time,
    ROW_NUMBER() OVER (
      PARTITION BY regulation_id
      ORDER BY watermark_col DESC
    ) AS rn,
    watermark_col as created_ts
  FROM src
)
SELECT
  regulation_id,
  regulation_number,
  regulation_name,
  regulation_department,
  regulation_region,
  regulation_risk_level,
  regulation_compliance_status,
  regulation_date_raised,
  regulation_date_closed,
  regulation_owner,
  regulation_deadline_date,
  is_regulation_on_time,
  created_ts
FROM dedup
WHERE rn = 1
    """,

    "attribute_cols": ["regulation_id", "regulation_number", "regulation_name", "regulation_department", "regulation_region", "regulation_risk_level", "regulation_compliance_status", "regulation_date_raised", "regulation_date_closed", "regulation_owner", "regulation_deadline_date", "is_regulation_on_time"
    ],

    "insert_cols": [
        "regulation_id",
        "regulation_number", 
        "regulation_name", 
        "regulation_department", 
        "regulation_region", 
        "regulation_risk_level", 
        "regulation_compliance_status", 
        "regulation_date_raised", 
        "regulation_date_closed", 
        "regulation_owner", 
        "regulation_deadline_date", 
        "is_regulation_on_time",
        "attr_hash",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_ACCOUNT_CFG = {
    "name": "dim_account",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_account",
    "business_key_col": "account_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            ha.account_id AS account_id,
            sa.account_number AS account_number,
            sa.account_type AS account_type,
            sa.account_status AS account_status,
            sa.account_creation_type AS account_creation_type,
            sa.account_last_access AS account_last_access_ts,
            sa.account_last_change AS account_last_change_ts,
            COALESCE(sa.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sa.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_account ha
          LEFT JOIN {catalog_name}.{vault_schema}.sat_account sa
            ON sa.account_hash_key = ha.account_hash_key
          WHERE sa.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            account_id,
            account_number,
            account_type,
            account_status,
            account_creation_type,
            account_last_access_ts,
            account_last_change_ts,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY account_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          account_id,
          account_number,
          account_type,
          account_status,
          account_creation_type,
          account_last_access_ts,
          account_last_change_ts,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "account_id",
        "account_number",
        "account_type",
        "account_status",
        "account_creation_type",
        "account_last_access_ts",
        "account_last_change_ts",
    ],

    "insert_cols": [
        "account_id",
        "account_number",
        "account_type",
        "account_status",
        "account_creation_type",
        "account_last_access_ts",
        "account_last_change_ts",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}


In [0]:
DIM_BROKER_CFG = {
    "name": "dim_broker",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_broker",
    "business_key_col": "agent_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hb.agent_id AS agent_id,
            sb.agent_name AS agent_name,
            sb.agent_type AS agent_type,
            sb.agent_status AS agent_status,
            sb.agent_license_number AS agent_license_number,
            sb.agent_net_promoter_score AS agent_net_promoter_score,
            sb.agent_commission_percentage AS agent_commission_percentage,
            COALESCE(sb.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sb.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_broker hb
          LEFT JOIN {catalog_name}.{vault_schema}.sat_broker sb
            ON hb.broker_hash_key = sb.broker_hash_key
          WHERE sb.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            agent_id,
            agent_name,
            agent_type,
            agent_status,
            agent_license_number,
            agent_net_promoter_score,
            agent_commission_percentage,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY agent_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          agent_id,
          agent_name,
          agent_type,
          agent_status,
          agent_license_number,
          agent_net_promoter_score,
          agent_commission_percentage,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "agent_id",
        "agent_name",
        "agent_type",
        "agent_status",
        "agent_license_number",
        "agent_net_promoter_score",
        "agent_commission_percentage",
    ],

    "insert_cols": [
        "agent_id",
        "agent_name",
        "agent_type",
        "agent_status",
        "agent_license_number",
        "agent_net_promoter_score",
        "agent_commission_percentage",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_CLAIM_CFG = {
    "name": "dim_claim",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_claim",
    "business_key_col": "claim_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hc.claim_id AS claim_id,
            sc.claim_number AS claim_number,
            sc.claim_type AS claim_type,
            sc.claim_status AS claim_status,
            sc.claim_reason AS claim_reason,
            sc.claim_channel AS claim_channel,
            sc.claim_handler AS claim_handler,
            sc.claim_reported_date AS claim_reported_date,
            sc.claim_settlement_date AS claim_settlement_date,
            sc.claim_product AS claim_product,
            sc.is_claim_suspicious AS is_claim_suspicious,
            sc.is_claim_fraud AS is_claim_fraud,
            sc.claim_fraud_status AS claim_fraud_status,
            sc.claim_fraud_type AS claim_fraud_type,
            sc.claim_fraud_detection_method AS claim_fraud_detection_method,
            sc.is_litigation AS is_litigation,
            sc.litigation_reason AS litigation_reason,
            sc.litigation_start_date AS litigation_start_date,
            sc.litigation_end_date AS litigation_end_date,
            sc.litigation_outcome AS litigation_outcome,
            sc.litigation_duration_days AS litigation_duration_days,
            sc.claim_fraud_detection_time_in_days AS claim_fraud_detection_time_in_days,
            sc.is_recovery_opportunity AS is_recovery_opportunity,
            sc.recovery_priority_score AS recovery_priority_score,
            sc.recovery_category AS recovery_category,
            sc.recovery_source AS recovery_source,
            sc.first_recovery_date AS first_recovery_date,
            sc.last_recovery_date AS last_recovery_date,
            sc.is_recovery_happened AS is_recovery_happened,
            sc.days_to_first_recovery AS days_to_first_recovery,
            sc.days_to_last_recovery AS days_to_last_recovery,
            sc.avg_days_to_close_claim AS avg_days_to_close_claim,
            sc.claim_fraud_outcome AS claim_fraud_outcome,
            sc.recovery_type AS recovery_type,
            sc.recovery_band AS recovery_band,
            sc.third_party_involved AS third_party_involved,
            sc.third_party_involved_overall_score AS third_party_involved_overall_score,
            sc.solicitor AS solicitor,
            sc.claim_amount AS claim_amt,
            sc.claims_paid AS claims_paid,
            sc.outstanding_reserve AS outstanding_reserve,
            sc.claims_expenses AS claims_expenses,
            sc.recovery_received AS recovery_received,
            sc.compensation_offered AS compensation_offered,
            sc.remediation_amount AS remediation_amt,
            sc.suspected_amount AS suspected_amt,
            sc.fraud_amount AS fraud_amt,
            sc.legal_expenses AS legal_expenses,
            sc.claim_band AS claim_band,
            sc.claim_band_sort AS claim_band_sort,
            sc.is_fault_claim AS is_fault_claim,
            sc.claim_satisfaction_score AS claim_satisfaction_score,
            sc.claims_feedback AS claims_feedback,
            COALESCE(sc.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sc.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_claim hc
          LEFT JOIN {catalog_name}.{vault_schema}.sat_claim sc
            ON hc.claim_hash_key = sc.claim_hash_key
          WHERE sc.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            claim_id,
            claim_number,
            claim_type,
            claim_status,
            claim_reason,
            claim_channel,
            claim_handler,
            claim_reported_date,
            claim_settlement_date,
            claim_product,
            is_claim_suspicious,
            is_claim_fraud,
            claim_fraud_status,
            claim_fraud_type,
            claim_fraud_detection_method,
            is_litigation,
            litigation_reason,
            litigation_start_date,
            litigation_end_date,
            litigation_outcome,
            litigation_duration_days,
            claim_fraud_detection_time_in_days,
            is_recovery_opportunity,
            recovery_priority_score,
            recovery_category,
            recovery_source,
            first_recovery_date,
            last_recovery_date,
            is_recovery_happened,
            days_to_first_recovery,
            days_to_last_recovery,
            avg_days_to_close_claim,
            claim_fraud_outcome,
            recovery_type,
            recovery_band,
            third_party_involved,
            third_party_involved_overall_score,
            solicitor,
            claim_amt,
            claims_paid,
            outstanding_reserve,
            claims_expenses,
            recovery_received,
            compensation_offered,
            remediation_amt,
            suspected_amt,
            fraud_amt,
            legal_expenses,
            claim_band,
            claim_band_sort,
            is_fault_claim,
            claim_satisfaction_score,
            claims_feedback,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY claim_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          claim_id,
          claim_number,
          claim_type,
          claim_status,
          claim_reason,
          claim_channel,
          claim_handler,
          claim_reported_date,
          claim_settlement_date,
          claim_product,
          is_claim_suspicious,
          is_claim_fraud,
          claim_fraud_status,
          claim_fraud_type,
          claim_fraud_detection_method,
          is_litigation,
          litigation_reason,
          litigation_start_date,
          litigation_end_date,
          litigation_outcome,
          litigation_duration_days,
          claim_fraud_detection_time_in_days,
          is_recovery_opportunity,
          recovery_priority_score,
          recovery_category,
          recovery_source,
          first_recovery_date,
          last_recovery_date,
          is_recovery_happened,
          days_to_first_recovery,
          days_to_last_recovery,
          avg_days_to_close_claim,
          claim_fraud_outcome,
          recovery_type,
          recovery_band,
          third_party_involved,
          third_party_involved_overall_score,
          solicitor,
          claim_amt,
          claims_paid,
          outstanding_reserve,
          claims_expenses,
          recovery_received,
          compensation_offered,
          remediation_amt,
          suspected_amt,
          fraud_amt,
          legal_expenses,
          claim_band,
          claim_band_sort,
          is_fault_claim,
          claim_satisfaction_score,
          claims_feedback,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "claim_id",
        "claim_number",
        "claim_type",
        "claim_status",
        "claim_reason",
        "claim_channel",
        "claim_handler",
        "claim_reported_date",
        "claim_settlement_date",
        "claim_product",
        "is_claim_suspicious",
        "is_claim_fraud",
        "claim_fraud_status",
        "claim_fraud_type",
        "claim_fraud_detection_method",
        "is_litigation",
        "litigation_reason",
        "litigation_start_date",
        "litigation_end_date",
        "litigation_outcome",
        "litigation_duration_days",
        "claim_fraud_detection_time_in_days",
        "is_recovery_opportunity",
        "recovery_priority_score",
        "recovery_category",
        "recovery_source",
        "first_recovery_date",
        "last_recovery_date",
        "is_recovery_happened",
        "days_to_first_recovery",
        "days_to_last_recovery",
        "avg_days_to_close_claim",
        "claim_fraud_outcome",
        "recovery_type",
        "recovery_band",
        "third_party_involved",
        "third_party_involved_overall_score",
        "solicitor",
        "claim_amt",
        "claims_paid",
        "outstanding_reserve",
        "claims_expenses",
        "recovery_received",
        "compensation_offered",
        "remediation_amt",
        "suspected_amt",
        "fraud_amt",
        "legal_expenses",
        "claim_band",
        "claim_band_sort",
        "is_fault_claim",
        "claim_satisfaction_score",
        "claims_feedback",
    ],

    "insert_cols": [
        "claim_id",
        "claim_number",
        "claim_type",
        "claim_status",
        "claim_reason",
        "claim_channel",
        "claim_handler",
        "claim_reported_date",
        "claim_settlement_date",
        "claim_product",
        "is_claim_suspicious",
        "is_claim_fraud",
        "claim_fraud_status",
        "claim_fraud_type",
        "claim_fraud_detection_method",
        "is_litigation",
        "litigation_reason",
        "litigation_start_date",
        "litigation_end_date",
        "litigation_outcome",
        "litigation_duration_days",
        "claim_fraud_detection_time_in_days",
        "is_recovery_opportunity",
        "recovery_priority_score",
        "recovery_category",
        "recovery_source",
        "first_recovery_date",
        "last_recovery_date",
        "is_recovery_happened",
        "days_to_first_recovery",
        "days_to_last_recovery",
        "avg_days_to_close_claim",
        "claim_fraud_outcome",
        "recovery_type",
        "recovery_band",
        "third_party_involved",
        "third_party_involved_overall_score",
        "solicitor",
        "claim_amt",
        "claims_paid",
        "outstanding_reserve",
        "claims_expenses",
        "recovery_received",
        "compensation_offered",
        "remediation_amt",
        "suspected_amt",
        "fraud_amt",
        "legal_expenses",
        "claim_band",
        "claim_band_sort",
        "is_fault_claim",
        "claim_satisfaction_score",
        "claims_feedback",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_CUSTOMER_CFG = {
    "name": "dim_customer",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_customer",
    "business_key_col": "customer_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hc.customer_id AS customer_id,
            sc.customer_number AS customer_number,
            sc.customer_rating AS customer_rating,
            sc.customer_segment AS customer_segment,
            sc.line_of_business AS line_of_business,
            sc.nps_score AS net_promoter_score,
            sc.customer_since AS customer_since_date,
            sc.customer_status AS customer_status_code,
            sc.customer_status_reason AS customer_status_reason,
            sc.income_band AS income_band,
            sc.customer_satisfaction AS customer_satisfaction,
            sc.customer_age_band AS customer_age_band,
            sc.net_promotor_code_segment AS net_promotor_code_segment,
			      sc.customer_onboarding_satisfaction_score AS customer_onboarding_satisfaction_score,
            sc.customer_onboarding_feedback AS customer_onboarding_feedback,
            COALESCE(sc.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sc.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_customer hc
          LEFT JOIN {catalog_name}.{vault_schema}.sat_customer sc
            ON hc.customer_hash_key = sc.customer_hash_key
          WHERE sc.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            customer_id,
            customer_number,
            customer_rating,
            customer_segment,
            line_of_business,
            net_promoter_score,
            customer_since_date,
            customer_status_code,
            customer_status_reason,
            income_band,
            customer_satisfaction,
            customer_age_band,
            net_promotor_code_segment,
			      customer_onboarding_satisfaction_score,
            customer_onboarding_feedback,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY customer_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          customer_id,
          customer_number,
          customer_rating,
          customer_segment,
          line_of_business,
          net_promoter_score,
          customer_since_date,
          customer_status_code,
          customer_status_reason,
          income_band,
          customer_satisfaction,
          customer_age_band,
          net_promotor_code_segment,
		      customer_onboarding_satisfaction_score,
          customer_onboarding_feedback,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "customer_id",
        "customer_number",
        "customer_rating",
        "customer_segment",
        "line_of_business",
        "net_promoter_score",
        "customer_since_date",
        "customer_status_code",
        "customer_status_reason",
        "income_band",
        "customer_satisfaction",
        "customer_age_band",
        "net_promotor_code_segment",
		    "customer_onboarding_satisfaction_score",
        "customer_onboarding_feedback"
    ],

    "insert_cols": [
        "customer_id",
        "customer_number",
        "customer_rating",
        "customer_segment",
        "line_of_business",
        "net_promoter_score",
        "customer_since_date",
        "customer_status_code",
        "customer_status_reason",
        "income_band",
        "customer_satisfaction",
        "customer_age_band",
        "net_promotor_code_segment",
		    "customer_onboarding_satisfaction_score",
        "customer_onboarding_feedback",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_HOME_CFG = {
    "name": "dim_home",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_home",
    "business_key_col": "insured_object_home_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
                  dio.insured_object_sk AS insured_object_sk,
                  hh.insured_object_home_id AS insured_object_home_id,
                  CASE
                  WHEN UPPER(TRIM(sh.is_existing_home_customer)) IN ('Y', 'YES', 'TRUE', '1') THEN'Y'
                  ELSE'N'
                  END AS is_existing_home_customer,
                  sh.home_risk_address AS home_risk_address,
                  sh.home_state AS home_state,
                  sh.home_type AS home_type,
                  sh.roof_construction AS roof_construction_material_type,
                  sh.wall_construction AS wall_construction_material_type,
                  COALESCE(sh.load_date, TIMESTAMP('1900-01-01 00:00:00')) AS effective_from_ts,
                  sh.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_home hh
          LEFT JOIN {catalog_name}.{vault_schema}.sat_home sh
          ON sh.home_hash_key = hh.home_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.link_insured_object_home lioh
          ON hh.home_hash_key = lioh.home_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hio
          ON lioh.insured_object_hash_key = hio.insured_object_hash_key
          LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object dio
          ON dio.insured_object_id = hio.insured_object_id
          WHERE sh.${{watermark_col}} > ${{watermark_col}}
          ),
          dedup AS (
          SELECT
                  insured_object_sk,
                  insured_object_home_id,
                  is_existing_home_customer,
                  home_risk_address,
                  home_state,
                  home_type,
                  roof_construction_material_type,
                  wall_construction_material_type,
                  effective_from_ts,
                  watermark_col,
                  ROW_NUMBER() OVER (
                      PARTITION BY insured_object_home_id
          ORDER BY watermark_col DESC
                  ) AS rn
          FROM src
          )
          SELECT
              insured_object_sk,
              insured_object_home_id,
              is_existing_home_customer,
              home_risk_address,
              home_state,
              home_type,
              roof_construction_material_type,
              wall_construction_material_type,
              watermark_col AS effective_from_ts,
              TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
          FROM dedup
          WHERE rn =1
    """,

    "attribute_cols": [
        "insured_object_sk",
        "insured_object_home_id",
        "is_existing_home_customer",
        "home_risk_address",
        "home_state",
        "home_type",
        "roof_construction_material_type",
        "wall_construction_material_type",
    ],

    "insert_cols": [
        "insured_object_sk",
        "insured_object_home_id",
        "is_existing_home_customer",
        "home_risk_address",
        "home_state",
        "home_type",
        "roof_construction_material_type",
        "wall_construction_material_type",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_IDENTITY_CFG = {
    "name": "dim_identity",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_identity",
    "business_key_col": "identity_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hi.identities_id AS identity_id,
            si.ecid AS experience_cloud_id,
            si.hashed_email AS email_address_hash_value,
            COALESCE(si.load_date, DATE('1900-01-01')) AS effective_from_ts,
            si.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_identities hi
          LEFT JOIN {catalog_name}.{vault_schema}.sat_identities si
            ON hi.identities_hash_key = si.identities_hash_key
          WHERE si.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            identity_id,
            experience_cloud_id,
            email_address_hash_value,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY identity_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          identity_id,
          experience_cloud_id,
          email_address_hash_value,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "identity_id",
        "experience_cloud_id",
        "email_address_hash_value",
    ],

    "insert_cols": [
        "identity_id",
        "experience_cloud_id",
        "email_address_hash_value",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_MARKETING_CFG = {
    "name": "dim_marketing",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_marketing",
    "business_key_cols": ["marketing_preference_id","marketing_engagement_id"],
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            mp.marketing_preference_id AS marketing_preference_id,
            lp.any AS is_any_communication,
            ll.preferred_contact_method AS preferred_contact_method,
            lp.email_subscriptions AS is_email_subscriptions,
            lp.commercial_email AS is_commercial_email,
            lp.email AS is_personal_email,
            lp.call AS is_call,
            lp.sms AS is_sms,
            lp.postal_mail AS is_postal_mail,
            me.marketing_engagement_id AS marketing_engagement_id,
            le.opened_email AS is_opened_email,
            le.marketing_status AS marketing_status,
            le.promotion_code AS promotion_code,
            le.has_retention_team_interaction AS has_retention_team_interaction,
            le.customer_service_call_frequency AS customer_service_call_frequency,
            le.average_call_sentiment AS average_call_sentiment,
            le.engagement_score AS engagement_score,
			      le.first_contact_resolution AS first_contact_resolution,
            greatest(
              COALESCE(lp.load_date, DATE('1900-01-01')),
              COALESCE(le.load_date, DATE('1900-01-01'))
            ) AS effective_from_ts,
            greatest(
              COALESCE(lp.${{watermark_col}}, DATE('1900-01-01')),
              COALESCE(le.${{watermark_col}}, DATE('1900-01-01'))
            ) AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_person hp
          LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_preference lpmp
            ON lpmp.person_hash_key = hp.person_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_preference mp
            ON mp.marketing_preference_hash_key = lpmp.marketing_preference_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.sat_marketing_preference lp
            ON lp.marketing_preference_hash_key = lpmp.marketing_preference_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.link_person_lead lpl
            ON lpl.person_hash_key = hp.person_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.sat_lead ll
            ON ll.lead_hash_key = lpl.lead_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_engagement lpme
            ON lpme.person_hash_key = hp.person_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_engagement me
            ON me.marketing_engagement_hash_key = lpme.marketing_engagement_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.sat_marketing_engagement le
            ON le.marketing_engagement_hash_key = lpme.marketing_engagement_hash_key
          WHERE greatest(
                  COALESCE(lp.${{watermark_col}}, DATE('1900-01-01')),
                  COALESCE(le.${{watermark_col}}, DATE('1900-01-01'))
                ) > ${{watermark_col}}
        ),
        business_key_cte AS (
          SELECT
            marketing_preference_id,
            marketing_engagement_id,
            is_any_communication,
            preferred_contact_method,
            is_email_subscriptions,
            is_commercial_email,
            is_personal_email,
            is_call,
            is_sms,
            is_postal_mail,
            is_opened_email,
            marketing_status,
            promotion_code,
            has_retention_team_interaction,
            customer_service_call_frequency,
            average_call_sentiment,
			      first_contact_resolution,
            engagement_score,
            effective_from_ts,
            watermark_col
          FROM src
        ),
        dedup AS (
          SELECT
            marketing_preference_id,
            marketing_engagement_id,
            is_any_communication,
            preferred_contact_method,
            is_email_subscriptions,
            is_commercial_email,
            is_personal_email,
            is_call,
            is_sms,
            is_postal_mail,
            is_opened_email,
            marketing_status,
            promotion_code,
            has_retention_team_interaction,
            customer_service_call_frequency,
            average_call_sentiment,
            engagement_score,
			first_contact_resolution,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY marketing_preference_id, marketing_engagement_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM business_key_cte
        )
        SELECT
          marketing_preference_id,
          marketing_engagement_id,
          is_any_communication,
          preferred_contact_method,
          is_email_subscriptions,
          is_commercial_email,
          is_personal_email,
          is_call,
          is_sms,
          is_postal_mail,
          is_opened_email,
          marketing_status,
          promotion_code,
          has_retention_team_interaction,
          customer_service_call_frequency,
          average_call_sentiment,
          engagement_score,
		      first_contact_resolution,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "marketing_preference_id",
        "is_any_communication",
        "preferred_contact_method",
        "is_email_subscriptions",
        "is_commercial_email",
        "is_personal_email",
        "is_call",
        "is_sms",
        "is_postal_mail",
        "marketing_engagement_id",
        "is_opened_email",
        "marketing_status",
        "promotion_code",
        "has_retention_team_interaction",
        "customer_service_call_frequency",
        "average_call_sentiment",
        "engagement_score",
		    "first_contact_resolution"
    ],

    "insert_cols": [
        "marketing_preference_id",
        "is_any_communication",
        "preferred_contact_method",
        "is_email_subscriptions",
        "is_commercial_email",
        "is_personal_email",
        "is_call",
        "is_sms",
        "is_postal_mail",
        "marketing_engagement_id",
        "is_opened_email",
        "marketing_status",
        "promotion_code",
        "has_retention_team_interaction",
        "customer_service_call_frequency",
        "average_call_sentiment",
        "engagement_score",
		    "first_contact_resolution",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_MOTOR_CFG = {
    "name": "dim_motor",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_motor",
    "business_key_col": "insured_object_motor_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            dm.insured_object_sk AS insured_object_sk,
            hm.insured_object_motor_id AS insured_object_motor_id,
            sm.auto_decline_vehicle AS is_auto_decline_vehicle,
            sm.is_existing_motor_customer AS is_existing_motor_customer,
            sm.motor_lapsed_policies AS motor_lapsed_policies,
            sm.motor_sum_insrd AS vehicle_sum_insured_amt,
            sm.risk_class_code AS vehicle_risk_class_code,
            sm.motor_risk_address AS vehicle_risk_address,
            sm.body_type AS vehicle_body_type,
            sm.fuel_type AS vehicle_fuel_type,
            sm.variant AS vehicle_variant,
            sm.vehicle_age AS vehicle_age,
            sm.vehicle_class AS vehicle_class,
            sm.vehicle_model AS vehicle_model,
            sm.vehicle_owner_type AS vehicle_owner_type,
            sm.vehicle_regstate AS vehicle_reg_state,
            sm.vehicle_type AS vehicle_type,
            sm.vehicle_year AS vehicle_year,
            sm.license_status AS driver_license_status,
            sm.driver_experience_years AS driver_experience_years,
            COALESCE(sm.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sm.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_motor hm
          LEFT JOIN {catalog_name}.{vault_schema}.link_insured_object_motor riom
            ON riom.motor_hash_key = hm.motor_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hio
            ON hio.insured_object_hash_key = riom.insured_object_hash_key
          LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object dm
            ON dm.insured_object_id = hio.insured_object_id
          LEFT JOIN {catalog_name}.{vault_schema}.sat_motor sm
            ON sm.motor_hash_key = hm.motor_hash_key
          WHERE sm.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            insured_object_sk,
            insured_object_motor_id,
            is_auto_decline_vehicle,
            is_existing_motor_customer,
            motor_lapsed_policies,
            vehicle_sum_insured_amt,
            vehicle_risk_class_code,
            vehicle_risk_address,
            vehicle_body_type,
            vehicle_fuel_type,
            vehicle_variant,
            vehicle_age,
            vehicle_class,
            vehicle_model,
            vehicle_owner_type,
            vehicle_reg_state,
            vehicle_type,
            vehicle_year,
            driver_license_status,
            driver_experience_years,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY insured_object_motor_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          insured_object_sk,
          insured_object_motor_id,
          is_auto_decline_vehicle,
          is_existing_motor_customer,
          motor_lapsed_policies,
          vehicle_sum_insured_amt,
          vehicle_risk_class_code,
          vehicle_risk_address,
          vehicle_body_type,
          vehicle_fuel_type,
          vehicle_variant,
          vehicle_age,
          vehicle_class,
          vehicle_model,
          vehicle_owner_type,
          vehicle_reg_state,
          vehicle_type,
          vehicle_year,
          driver_license_status,
          driver_experience_years,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "insured_object_sk",
        "insured_object_motor_id",
        "is_auto_decline_vehicle",
        "is_existing_motor_customer",
        "motor_lapsed_policies",
        "vehicle_sum_insured_amt",
        "vehicle_risk_class_code",
        "vehicle_risk_address",
        "vehicle_body_type",
        "vehicle_fuel_type",
        "vehicle_variant",
        "vehicle_age",
        "vehicle_class",
        "vehicle_model",
        "vehicle_owner_type",
        "vehicle_reg_state",
        "vehicle_type",
        "vehicle_year",
        "driver_license_status",
        "driver_experience_years",
    ],

    "insert_cols": [
        "insured_object_sk",
        "insured_object_motor_id",
        "is_auto_decline_vehicle",
        "is_existing_motor_customer",
        "motor_lapsed_policies",
        "vehicle_sum_insured_amt",
        "vehicle_risk_class_code",
        "vehicle_risk_address",
        "vehicle_body_type",
        "vehicle_fuel_type",
        "vehicle_variant",
        "vehicle_age",
        "vehicle_class",
        "vehicle_model",
        "vehicle_owner_type",
        "vehicle_reg_state",
        "vehicle_type",
        "vehicle_year",
        "driver_license_status",
        "driver_experience_years",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_PERSON_CFG = {
    "name": "dim_person",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_person",
    "business_key_col": "person_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
    WITH hub_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_person
    ),

    link_person_natural_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_natural_person
    ),

    hub_natural_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_natural_person
    ),

    sat_natural_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_natural_person
    ),

    link_person_legal_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_legal_person
    ),

    hub_legal_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_legal_person
    ),

    sat_legal_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_legal_person
    ),

    link_person_address AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_address
    ),

    hub_address AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_address
    ),

    sat_address AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_address
    ),

    link_person_contact AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_contact
    ),

    hub_contact AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_contact
    ),

    sat_contact AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_contact
    ),

    link_person_consent AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_consent
    ),

    hub_consent AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_consent
    ),

    sat_consent AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_consent
    ),

    link_person_identities AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.link_person_identities
    ),

    hub_identities AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.hub_identities
    ),

    sat_identities AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_identities
    ),

    sat_person AS (
        SELECT *
        FROM {catalog_name}.{vault_schema}.sat_person
    ),

    base_person AS (
        SELECT
            dg.geography_sk AS geography_sk,
            di.identity_sk AS identity_sk,
            COALESCE(
                hp.person_id,
                hnp.natural_person_id,
                hlp.legal_person_id
            ) AS person_id,

            snp.courtesy_title AS courtesy_title,
            snp.first_name AS first_name,
            snp.last_name AS last_name,
            snp.full_name AS full_name,
            sp.type AS person_type,
            snp.birth_date AS birth_date,
            snp.gender AS gender,
            snp.nationality AS nationality,
            snp.marital_status AS marital_status,
            snp.occupation AS occupation,
            ha.address_id AS address_id,
            sa.type AS address_type,
            sa.street AS street_address,
            sa.postcode AS postcode,
            hc.contact_id AS contact_id,
            sc.home_phone AS home_phone_number,
            sc.work_phone AS work_phone_number,
            sc.personal_email AS personal_email,
            sc.work_email AS work_email,
            snp.job_title AS job_title,
            snp.role AS role,
            sl.company_name AS company_name,
            sl.date_of_constitution AS date_of_constitution,
            snp.preferred_language,
            snp.assesed_disability_degree,
            hco.consent_id,
			sp.source_id,
            sp.source_type,
            sp.tenant_id,
            sp.is_lead,
            sp.operational_paperless_consent,
            sco.opt_in_legitimate_interest AS is_opt_in_legitimate_interest,
            sco.opt_in_validated AS is_opt_in_validated,

            greatest(
                COALESCE(sp.load_date, DATE('1900-01-01')),
                COALESCE(snp.load_date, DATE('1900-01-01')),
                COALESCE(sl.load_date, DATE('1900-01-01')),
                COALESCE(sc.load_date, DATE('1900-01-01')),
                COALESCE(sa.load_date, DATE('1900-01-01')),
                COALESCE(sco.load_date, DATE('1900-01-01'))
            ) AS effective_from_ts,
            greatest(
                COALESCE(sp.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(snp.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sl.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sc.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sa.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sco.${{watermark_col}}, DATE('1900-01-01'))
            ) AS watermark_col

        FROM hub_person hp

        LEFT JOIN link_person_natural_person lpn
            ON lpn.person_hash_key = hp.person_hash_key

        LEFT JOIN hub_natural_person hnp
            ON hnp.natural_person_hash_key = lpn.natural_person_hash_key

        LEFT JOIN sat_natural_person snp
            ON snp.natural_person_hash_key = lpn.natural_person_hash_key

        LEFT JOIN link_person_legal_person lpl
            ON lpl.person_hash_key = hp.person_hash_key

        LEFT JOIN hub_legal_person hlp
            ON hlp.legal_person_hash_key = lpl.legal_person_hash_key

        LEFT JOIN sat_legal_person sl
            ON sl.legal_person_hash_key = lpl.legal_person_hash_key

        LEFT JOIN link_person_address lha
            ON lha.person_hash_key = hp.person_hash_key

        LEFT JOIN hub_address ha
            ON ha.address_hash_key = lha.address_hash_key

        LEFT JOIN sat_address sa
            ON sa.address_hash_key = lha.address_hash_key

        LEFT JOIN {catalog_name}.{gold_schema_name}.dim_geography dg
            ON dg.city = sa.city
            AND dg.state = sa.state
            AND dg.country = sa.country

        LEFT JOIN link_person_contact lpc
            ON lpc.person_hash_key = hp.person_hash_key

        LEFT JOIN hub_contact hc
            ON hc.contact_hash_key = lpc.contact_hash_key

        LEFT JOIN sat_contact sc
            ON sc.contact_hash_key = lpc.contact_hash_key

        LEFT JOIN {catalog_name}.{gold_schema_name}.dim_identity di
            ON md5(sc.personal_email) = di.email_address_hash_value

        LEFT JOIN link_person_consent lpco
            ON lpco.person_hash_key = hp.person_hash_key

        LEFT JOIN hub_consent hco
            ON hco.consent_hash_key = lpco.consent_hash_key

        LEFT JOIN sat_consent sco
            ON sco.consent_hash_key = lpco.consent_hash_key

        LEFT JOIN sat_person sp
            ON sp.person_hash_key = hp.person_hash_key

        WHERE greatest(
                COALESCE(sp.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(snp.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sl.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sc.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sa.${{watermark_col}}, DATE('1900-01-01')),
                COALESCE(sco.${{watermark_col}}, DATE('1900-01-01'))
            ) > ${{watermark_col}}
    ),

    deduped AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY person_id
                ORDER BY effective_from_ts DESC
            ) AS rn
        FROM base_person
    )

    SELECT
        geography_sk,
        identity_sk,
        person_id,
        courtesy_title,
        first_name,
        last_name,
        full_name,
        person_type,
        birth_date,
        gender,
        nationality,
        marital_status,
        occupation,
        address_id,
        address_type,
        street_address,
        postcode,
        contact_id,
        home_phone_number,
        work_phone_number,
        personal_email,
        work_email,
        job_title,
        role,
        company_name,
        date_of_constitution,
        preferred_language,
        assesed_disability_degree as assessed_disability_degree,
        consent_id,
        source_id,
        source_type,
        tenant_id,
        is_lead,
        operational_paperless_consent as is_operational_paperless_consent,
        is_opt_in_legitimate_interest,
        is_opt_in_validated, 
        watermark_col AS effective_from_ts,
        TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts


    FROM deduped
    WHERE rn = 1
    """,

    "attribute_cols": [
        "geography_sk",
        "identity_sk",
        "person_id",
        "courtesy_title",
        "first_name",
        "last_name",
        "full_name",
        "person_type",
        "birth_date",
        "gender",
        "nationality",
        "marital_status",
        "occupation",
        "address_id",
        "address_type",
        "street_address",
        "postcode",
        "contact_id",
        "home_phone_number",
        "work_phone_number",
        "personal_email",
        "work_email",
        "job_title",
        "role",
        "company_name",
        "date_of_constitution",
        "preferred_language",
        "assessed_disability_degree",
        "consent_id",
        "source_id",
		"source_type",
        "tenant_id",
        "is_lead",
        "is_operational_paperless_consent",
        "is_opt_in_legitimate_interest",
        "is_opt_in_validated"  
    ],

    "insert_cols": [
        "geography_sk",
        "identity_sk",
        "person_id",
        "courtesy_title",
        "first_name",
        "last_name",
        "full_name",
        "person_type",
        "birth_date",
        "gender",
        "nationality",
        "marital_status",
        "occupation",
        "address_id",
        "address_type",
        "street_address",
        "postcode",
        "contact_id",
        "home_phone_number",
        "work_phone_number",
        "personal_email",
        "work_email",
        "job_title",
        "role",
        "company_name",
        "date_of_constitution",
        "preferred_language",
        "assessed_disability_degree",
        "consent_id",
        "source_id",
		"source_type",
        "tenant_id",
        "is_lead",
        "is_operational_paperless_consent",
        "is_opt_in_legitimate_interest",
        "is_opt_in_validated",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}


In [0]:
DIM_POLICY_CFG = {
    "name": "dim_policy",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_policy",
    "business_key_col": "policy_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hp.policy_id AS policy_id,
            sp.policy_number AS policy_number,
            sp.policy_start_date AS policy_start_ts,
            sp.policy_end_date AS policy_end_ts,
            sp.policy_length AS policy_tenure,
            sp.policy_cycle AS policy_cycle,
            sp.renewal_date AS renewal_date,
            sp.policy_status AS policy_status,
            sp.cover_option AS policy_cover_option,
            sp.sales_channel AS policy_sales_channel,
            sp.fraud_flag AS is_fraud,
            hq.quote_id AS quote_id,
            sp.policy_type AS policy_type,
            sp.policy_issue_date AS policy_issue_date,
            sp.is_policy_renewal AS is_policy_renewal,
            sp.policy_cancellation_reason AS policy_cancellation_reason,
            sp.policy_sum_insured AS policy_sum_insured,
            sp.policy_retention_limit AS policy_retention_limit,
            sp.policy_risk_score AS policy_risk_score,
            sp.policy_risk_band AS policy_risk_band,
            sp.is_auto_renew_enabled AS is_auto_renew_enabled,
            sp.no_claims_discount_years AS no_claims_discount_years,
            sp.payment_method AS payment_method,
            sp.is_direct_debit_cancellation AS is_direct_debit_cancellation,
            sp.missed_payment_count AS missed_payment_count,
            sp.loyalty_discount_usage AS loyalty_discount_usage,
            sp.is_installment_default AS is_installment_default,
            dc.channel_sk AS channel_sk,
            dp.product_sk AS product_sk,
            doi.insured_object_sk AS insured_object_sk,
            COALESCE(sp.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sp.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_policy hp
          LEFT JOIN {catalog_name}.{vault_schema}.sat_policy sp
            ON hp.policy_hash_key = sp.policy_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.link_policy_channel rpc
            ON rpc.policy_hash_key = hp.policy_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_channel hc
            ON hc.channel_hash_key = rpc.channel_hash_key
          LEFT JOIN {catalog_name}.{gold_schema_name}.dim_channel dc
            ON dc.channel_id = hc.channel_id
          LEFT JOIN {catalog_name}.{vault_schema}.link_policy_product lpp
            ON lpp.policy_hash_key = hp.policy_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_product hpr
            ON hpr.product_hash_key = lpp.product_hash_key
          LEFT JOIN {catalog_name}.{gold_schema_name}.dim_product dp
            ON dp.product_id = hpr.product_id
          LEFT JOIN {catalog_name}.{vault_schema}.link_policy_insured_object lio
            ON lio.policy_hash_key = hp.policy_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hi
            ON hi.insured_object_hash_key = lio.insured_object_hash_key
          LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object doi
            ON doi.insured_object_id = hi.insured_object_id
          LEFT JOIN {catalog_name}.{vault_schema}.link_policy_quote lq
            ON hp.policy_hash_key = lq.policy_hash_key
          LEFT JOIN {catalog_name}.{vault_schema}.hub_quote hq
            ON hq.quote_hash_key = lq.quote_hash_key
          WHERE sp.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            policy_id,
            policy_number,
            policy_start_ts,
            policy_end_ts,
            policy_tenure,
            policy_cycle,
            renewal_date,
            policy_status,
            policy_cover_option,
            policy_sales_channel,
            is_fraud,
            quote_id,
            policy_type,
            policy_issue_date,
            is_policy_renewal,
            policy_cancellation_reason,
            policy_sum_insured,
            policy_retention_limit,
            policy_risk_score,
            policy_risk_band,
            is_auto_renew_enabled,
            no_claims_discount_years,
            payment_method,
            is_direct_debit_cancellation,
            missed_payment_count,
            loyalty_discount_usage,
            is_installment_default,
            channel_sk,
            product_sk,
            COALESCE(insured_object_sk,-1) AS insured_object_sk,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY policy_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          policy_id,
          policy_number,
          policy_start_ts,
          policy_end_ts,
          policy_tenure,
          policy_cycle,
          renewal_date,
          policy_status,
          policy_cover_option,
          policy_sales_channel,
          is_fraud,
          quote_id,
          policy_type,
          policy_issue_date,
          is_policy_renewal,
          policy_cancellation_reason,
          policy_sum_insured,
          policy_retention_limit,
          policy_risk_score,
          policy_risk_band,
          is_auto_renew_enabled,
          no_claims_discount_years,
          payment_method,
          is_direct_debit_cancellation,
          missed_payment_count,
          loyalty_discount_usage,
          is_installment_default,
          channel_sk,
          product_sk,
          coalesce(insured_object_sk,-1) AS insured_object_sk,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "channel_sk",
        "product_sk",
        "insured_object_sk",
        "policy_id",
        "policy_number",
        "policy_start_ts",
        "policy_end_ts",
        "policy_tenure",
        "policy_cycle",
        "renewal_date",
        "policy_status",
        "policy_cover_option",
        "policy_sales_channel",
        "is_fraud",
        "quote_id",
        "policy_type",
        "policy_issue_date",
        "is_policy_renewal",
        "policy_cancellation_reason",
        "policy_sum_insured",
        "policy_retention_limit",
        "policy_risk_score",
        "policy_risk_band",
        "is_auto_renew_enabled",
        "no_claims_discount_years",
        "payment_method",
        "is_direct_debit_cancellation",
        "missed_payment_count",
        "loyalty_discount_usage",
        "is_installment_default",
    ],

    "insert_cols": [
        "channel_sk",
        "product_sk",
        "insured_object_sk",
        "policy_id",
        "policy_number",
        "policy_start_ts",
        "policy_end_ts",
        "policy_tenure",
        "policy_cycle",
        "renewal_date",
        "policy_status",
        "policy_cover_option",
        "policy_sales_channel",
        "is_fraud",
        "quote_id",
        "policy_type",
        "policy_issue_date",
        "is_policy_renewal",
        "policy_cancellation_reason",
        "policy_sum_insured",
        "policy_retention_limit",
        "policy_risk_score",
        "policy_risk_band",
        "is_auto_renew_enabled",
        "no_claims_discount_years",
        "payment_method",
        "is_direct_debit_cancellation",
        "missed_payment_count",
        "loyalty_discount_usage",
        "is_installment_default",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}

In [0]:
DIM_PRODUCT_CFG = {
    "name": "dim_product",
    "target_table": f"{catalog_name}.{gold_schema_name}.dim_product",
    "business_key_col": "product_id",
    "scd_type": "2",
    "watermark_col": "load_date",
    "effective_from_col": "effective_from_ts",
    "record_version_col": "record_version",

    "stage_sql": f"""
        WITH src AS (
          SELECT
            hp.product_id AS product_id,
            sp.type AS product_type,
            sp.product_variant AS product_variant,
            sp.product_name AS product_name,
            sp.product_launch_date AS product_launch_date,
            sp.product_status AS product_status,
            sp.product_line_of_business_code AS product_line_of_business_code,
            sp.underwriting_group AS underwriting_group,
            sp.regulatory_approval_code AS regulatory_approval_code,
            COALESCE(sp.load_date, DATE('1900-01-01')) AS effective_from_ts,
            sp.${{watermark_col}} AS watermark_col
          FROM {catalog_name}.{vault_schema}.hub_product hp
          LEFT JOIN {catalog_name}.{vault_schema}.sat_product sp
            ON hp.product_hash_key = sp.product_hash_key
          WHERE sp.${{watermark_col}} > ${{watermark_col}}
        ),
        dedup AS (
          SELECT
            product_id,
            product_type,
            product_variant,
            product_name,
            product_launch_date,
            product_status,
            product_line_of_business_code,
            underwriting_group,
            regulatory_approval_code,
            effective_from_ts,
            watermark_col,
            ROW_NUMBER() OVER (
              PARTITION BY product_id
              ORDER BY watermark_col DESC
            ) AS rn
          FROM src
        )
        SELECT
          product_id,
          product_type,
          product_variant,
          product_name,
          product_launch_date,
          product_status,
          product_line_of_business_code,
          underwriting_group,
          regulatory_approval_code,
          watermark_col AS effective_from_ts,
          TIMESTAMP '9999-12-31 00:00:00' AS effective_to_ts
        FROM dedup
        WHERE rn = 1
    """,

    "attribute_cols": [
        "product_id",
        "product_type",
        "product_variant",
        "product_name",
        "product_launch_date",
        "product_status",
        "product_line_of_business_code",
        "underwriting_group",
        "regulatory_approval_code",
    ],

    "insert_cols": [
        "product_id",
        "product_type",
        "product_variant",
        "product_name",
        "product_launch_date",
        "product_status",
        "product_line_of_business_code",
        "underwriting_group",
        "regulatory_approval_code",
        "attr_hash",
        "effective_from_ts",
        "effective_to_ts",
        "record_version",
        "created_by",
        "created_ts",
        "last_updated_by",
        "last_updated_ts",
    ]
}